# Classificação multiclasse de reviews em português

Este notebook reorganiza o fluxo do `main.py` em etapas didáticas e reproduzíveis:
1. instalar dependências;
2. carregar os dados;
3. preprocessar texto e sinais de produto;
4. construir features textuais e numéricas;
5. treinar e validar o modelo;
6. refazer o treino com 100% do conjunto de treino;
7. gerar `submission.csv`.

A estrutura foi baseada principalmente no fluxo principal do projeto, com apoio dos módulos auxiliares de preprocessamento, features, pipeline e avaliação.

## Como usar em outra máquina

Este notebook foi reorganizado para funcionar como um **roteiro completo de treinamento, avaliação e geração de submissão**.

### O que você precisa ter
- Python 3.10+ recomendado;
- os arquivos `train.csv`, `test.csv` e `sample_submission.csv`;
- as bibliotecas listadas em `requirements.txt`.

### Fluxo sugerido de uso
1. Coloque este notebook na mesma pasta dos CSVs.
2. Execute a célula de instalação/importação.
3. Revise a seção **Configuração** e ajuste caminhos ou hiperparâmetros.
4. Rode as células em ordem, observando as saídas intermediárias.
5. Ao final, confira o arquivo `submission.csv` gerado.

### O que foi feito para deixá-lo replicável
Em vez de depender diretamente da estrutura modular do projeto, as funções essenciais foram trazidas para dentro do notebook. Assim:
- ele pode ser compartilhado isoladamente;
- outras pessoas conseguem reproduzir o experimento com menos setup;
- a lógica do pipeline fica visível em um só lugar, o que facilita estudo e manutenção.

### Estrutura do notebook
- **Configuração:** define caminhos, modelo e parâmetros.
- **Funções utilitárias:** logging e inspeção rápida do dataset.
- **Preprocessamento:** limpeza textual e preparação de colunas auxiliares.
- **Feature engineering:** criação de atributos numéricos além do texto.
- **Pipeline:** vetorização + escalonamento + classificador.
- **Avaliação:** validação cruzada opcional e holdout.
- **Refit final:** treino em 100% do treino e geração da submissão.

> Dica: leia os blocos de markdown antes de executar as células de código. Eles explicam o motivo de cada etapa, e não apenas o que está sendo feito.


In [ ]:
# Instale as dependências se necessário.
# Em alguns ambientes (Kaggle, Colab, Jupyter local já configurado), talvez esta célula não seja necessária.

# %pip install -q pandas>=2.0.0 numpy>=1.24.0 scikit-learn>=1.3.0 lightgbm>=4.0.0

In [ ]:
from __future__ import annotations

import re
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC

## Configuração

Nesta seção você define os parâmetros principais do experimento.

### O que pode ser alterado
- **caminhos dos arquivos**: onde estão `train.csv`, `test.csv` e `sample_submission.csv`;
- **modelo**: por padrão, o notebook usa `linear_svm`, que costuma funcionar bem em classificação de textos esparsos;
- **features textuais**: tamanho máximo do vocabulário de palavras e de caracteres;
- **features numéricas**: modo `full` ou `minimal`;
- **validação**: número de folds da validação cruzada e tamanho do holdout;
- **reprodutibilidade**: `RANDOM_STATE`.

### Sobre as escolhas padrão
As escolhas iniciais tentam equilibrar:
- bom desempenho em texto;
- custo computacional moderado;
- facilidade de replicação em máquinas comuns.

### Parâmetros mais importantes
- `FEATURE_MODE="full"`: usa todas as features manuais;
- `FEATURE_MODE="minimal"`: usa apenas sinais mais robustos e compactos;
- `USE_CHAR_NGRAMS=True`: adiciona padrões de caracteres, úteis para textos ruidosos;
- `SVM_C`: controla a regularização do SVM;
- `SVM_CLASS_WEIGHT`: pode ajudar em caso de classes desbalanceadas.


In [ ]:
# =========================
# CONFIGURAÇÃO DO EXPERIMENTO
# =========================

TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"
SAMPLE_SUBMISSION_PATH = "sample_submission.csv"

TARGET_COLUMN = "rating"
SUBMISSION_PATH = "submission.csv"

MODEL_NAME = "linear_svm"          # opções: "linear_svm", "logreg", "lightgbm"
FEATURE_MODE = "full"              # opções: "full", "minimal"
USE_CHAR_NGRAMS = True
COUNT_MAX_FEATURES = 30_000
CHAR_MAX_FEATURES = 20_000

SVM_C = 10.0
SVM_CLASS_WEIGHT = "balanced"      # opções: "none", "balanced"

CV_FOLDS = 0                       # use 0 para desativar
TEST_SIZE = 0.20
RANDOM_STATE = 42

In [ ]:
# Função simples para padronizar mensagens do notebook com timestamp.
# Isso ajuda a acompanhar o fluxo de execução, especialmente em treinos mais longos.
def log(message: str) -> None:
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}] {message}")

## Funções utilitárias

A seguir concentramos funções reutilizáveis que seriam, em um projeto tradicional, distribuídas em arquivos separados.

A ideia aqui é deixar o notebook **autossuficiente** e, ao mesmo tempo, **legível**.

### Blocos principais de funções
1. **Preprocessamento**
   - limpa campos textuais;
   - valida colunas esperadas;
   - cria `combined_text`;
   - transforma `ASIN` em sinais úteis.

2. **Feature engineering**
   - cria estatísticas simples sobre o texto;
   - calcula sinais de polaridade lexical;
   - adiciona indicadores de temas recorrentes como defeito, preço e entrega.

3. **Pipeline e avaliação**
   - junta texto e atributos numéricos;
   - treina o classificador;
   - calcula métricas de validação.

> Mesmo quando uma função parece pequena, ela existe para manter a lógica organizada e facilitar testes e modificações futuras.


In [ ]:
# Conjunto de funções responsáveis por preparar os dados brutos antes do treinamento.
# Aqui validamos colunas, tratamos texto, criamos combined_text e derivamos sinais do ASIN.
# =========================
# PREPROCESSAMENTO
# =========================

BASE_REQUIRED_COLUMNS = ["id", "ASIN", "title", "text"]
TEST_REQUIRED_COLUMNS = ["id", "ASIN", "title", "text"]

PORTUGUESE_STOPWORDS = {
    "a", "o", "as", "os", "um", "uma", "uns", "umas",
    "de", "da", "do", "das", "dos",
    "em", "no", "na", "nos", "nas",
    "para", "por", "com", "sem",
    "e", "ou", "que", "como", "muito", "muita", "muitos", "muitas",
    "se", "isso", "essa", "esse", "esses", "essas", "isto", "aquilo",
    "foi", "ser", "ter", "sou", "era", "sao", "são", "é",
    "ao", "aos", "à", "às", "seu", "sua", "seus", "suas",
    "meu", "minha", "meus", "minhas", "você", "vocês", "eu", "nós",
}

ANTITHESIS_TERMS = [
    "mas",
    "porém",
    "porem",
    "entretanto",
    "contudo",
    "todavia",
    "apesar de",
    "não obstante",
    "nao obstante",
]

def normalize_text_for_vectorizer(text: str) -> str:
    normalized = str(text).lower()
    normalized = re.sub(r"http\\S+|www\\S+", " ", normalized)
    normalized = re.sub(r"\\s+", " ", normalized).strip()
    return normalized

def get_portuguese_stopwords() -> list[str]:
    antithesis_single_terms = {
        term.lower()
        for term in ANTITHESIS_TERMS
        if len(term.split()) == 1
    }
    filtered_stopwords = PORTUGUESE_STOPWORDS - antithesis_single_terms
    return sorted(filtered_stopwords)

def _validate_columns(df: pd.DataFrame, required_columns: list[str], file_label: str) -> None:
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise ValueError(f"{file_label} sem colunas obrigatórias: {missing}")

def load_train_test(train_path: str, test_path: str, target_column: str):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    train_required_columns = BASE_REQUIRED_COLUMNS + [target_column]
    _validate_columns(train_df, train_required_columns, "train file")
    _validate_columns(test_df, TEST_REQUIRED_COLUMNS, "test file")
    return train_df, test_df

def clean_text_fields(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()

    cleaned["title"] = cleaned["title"].fillna("").astype(str).str.strip()
    cleaned["text"] = cleaned["text"].fillna("").astype(str).str.strip()
    cleaned["ASIN"] = cleaned["ASIN"].fillna("UNKNOWN").astype(str).str.strip()

    cleaned["title"] = cleaned["title"].replace(".", "", regex=False)
    cleaned["combined_text"] = (cleaned["title"] + " " + cleaned["text"]).str.strip()
    cleaned["combined_text"] = cleaned["combined_text"].replace("", "sem comentario")

    return cleaned

def fit_asin_encoder(train_df: pd.DataFrame) -> LabelEncoder:
    encoder = LabelEncoder()
    encoder.fit(train_df["ASIN"].astype(str))
    return encoder

def transform_asin(df: pd.DataFrame, encoder: LabelEncoder) -> pd.Series:
    asin_to_int = {asin: idx for idx, asin in enumerate(encoder.classes_)}
    return df["ASIN"].map(asin_to_int).fillna(-1).astype(int)

def fit_asin_frequency(train_df: pd.DataFrame) -> pd.Series:
    return train_df["ASIN"].value_counts(normalize=True)

def transform_asin_frequency(df: pd.DataFrame, asin_frequency: pd.Series) -> pd.Series:
    return df["ASIN"].map(asin_frequency).fillna(0.0).astype(float)

def prepare_datasets(train_df: pd.DataFrame, test_df: pd.DataFrame):
    train_clean = clean_text_fields(train_df)
    test_clean = clean_text_fields(test_df)

    asin_encoder = fit_asin_encoder(train_clean)
    asin_frequency = fit_asin_frequency(train_clean)

    train_clean["asin_encoded"] = transform_asin(train_clean, asin_encoder)
    test_clean["asin_encoded"] = transform_asin(test_clean, asin_encoder)

    train_clean["asin_freq"] = transform_asin_frequency(train_clean, asin_frequency)
    test_clean["asin_freq"] = transform_asin_frequency(test_clean, asin_frequency)

    return train_clean, test_clean, asin_encoder

In [ ]:
# Neste bloco são criadas as features manuais.
# Elas complementam a vetorização do texto com sinais numéricos interpretáveis.
# =========================
# FEATURE ENGINEERING
# =========================

POSITIVE_WORDS = {
    "bom", "boa", "bons", "boas", "otimo", "otima", "ótimo", "ótima",
    "excelente", "recomendo", "gostei", "adorei", "perfeito", "perfeita",
    "maravilhoso", "maravilhosa", "amei", "satisfeito", "satisfeita",
    "qualidade", "beneficio", "benefício", "aprovado", "funciona",
    "funcionou", "duravel", "durável",
}

NEGATIVE_WORDS = {
    "ruim", "péssimo", "pessimo", "péssima", "pessima", "horrível", "horrivel",
    "decepcao", "decepção", "decepcionante", "nao", "não", "nunca", "absurdo",
    "rasgado", "quebrou", "defeito", "defeituoso", "fraco", "travando",
    "frustrante", "caro", "lento", "atraso", "demorou",
}

WORD_PATTERN = re.compile(r"\b\w+\b", flags=re.UNICODE)
TOKEN_PATTERN = re.compile(r"[A-Za-zÀ-ÿ]+", flags=re.UNICODE)
CAPS_PATTERN = re.compile(r"\b[A-ZÀ-Ý]{2,}\b", flags=re.UNICODE)

DEFECT_TERMS = [
    "quebrou", "rasgado", "defeito", "defeituoso", "faltando", "faltou",
    "nao funciona", "não funciona", "nao lig", "não lig", "travando", "deteriorado",
]

PRICE_TERMS = [
    "preco", "preço", "caro", "barato", "custo", "valor", "abusivo",
    "custo beneficio", "custo-beneficio", "custo-benefício", "nao vale", "não vale",
]

DELIVERY_TERMS = [
    "entrega", "entregue", "chegou", "prazo", "rastre", "postagem", "atras", "atraso", "demorou",
]

NEGATION_TERMS = {"nao", "não", "nunca", "nem", "jamais"}

def _word_count(text: str) -> int:
    return len(WORD_PATTERN.findall(text))

def _char_count(text: str) -> int:
    return len(text)

def _exclamation_count(text: str) -> int:
    return text.count("!")

def _question_count(text: str) -> int:
    return text.count("?")

def _caps_word_count(text: str) -> int:
    return len(CAPS_PATTERN.findall(text))

def _tokenize_lower(text: str) -> list[str]:
    return [token.lower() for token in TOKEN_PATTERN.findall(text)]

def _count_lexicon_hits(tokens: list[str], lexicon: set[str]) -> int:
    return sum(1 for token in tokens if token in lexicon)

def _count_pattern_occurrences(lowered: str, terms: list[str]) -> int:
    total = 0
    for term in terms:
        if " " in term:
            total += lowered.count(term)
        else:
            total += len(re.findall(rf"\b{re.escape(term)}", lowered))
    return total

def _sentiment_score(text: str) -> float:
    tokens = _tokenize_lower(text)
    if not tokens:
        return 0.0
    positive_hits = _count_lexicon_hits(tokens, POSITIVE_WORDS)
    negative_hits = _count_lexicon_hits(tokens, NEGATIVE_WORDS)
    return (positive_hits - negative_hits) / max(len(tokens), 1)

def _antithesis_norm_count(text: str, word_count: int) -> float:
    lowered = text.lower()
    occurrences = _count_pattern_occurrences(lowered, ANTITHESIS_TERMS)
    return occurrences / max(word_count, 1)

def build_feature_frame(df: pd.DataFrame) -> pd.DataFrame:
    features = pd.DataFrame(index=df.index)

    text_series = df["combined_text"].fillna("").astype(str)
    title_series = df.get("title", pd.Series("", index=df.index)).fillna("").astype(str)
    body_series = df.get("text", pd.Series("", index=df.index)).fillna("").astype(str)
    lowered_series = text_series.str.lower()
    token_series = text_series.apply(_tokenize_lower)

    features["combined_text"] = text_series

    features["word_count"] = text_series.apply(_word_count)
    features["char_count"] = text_series.apply(_char_count)
    features["exclamation_count"] = text_series.apply(_exclamation_count)
    features["question_count"] = text_series.apply(_question_count)
    features["caps_word_count"] = text_series.apply(_caps_word_count)

    features["sentiment_score"] = text_series.apply(_sentiment_score)
    features["positive_word_count"] = token_series.apply(lambda tokens: _count_lexicon_hits(tokens, POSITIVE_WORDS))
    features["negative_word_count"] = token_series.apply(lambda tokens: _count_lexicon_hits(tokens, NEGATIVE_WORDS))
    features["sentiment_ratio"] = (features["positive_word_count"] + 1.0) / (features["negative_word_count"] + 1.0)
    features["mixed_sentiment_flag"] = (
        (features["positive_word_count"] > 0) & (features["negative_word_count"] > 0)
    ).astype(int)

    features["defect_term_count"] = lowered_series.apply(lambda text: _count_pattern_occurrences(text, DEFECT_TERMS))
    features["defect_flag"] = (features["defect_term_count"] > 0).astype(int)

    features["price_term_count"] = lowered_series.apply(lambda text: _count_pattern_occurrences(text, PRICE_TERMS))
    features["price_flag"] = (features["price_term_count"] > 0).astype(int)

    features["delivery_term_count"] = lowered_series.apply(lambda text: _count_pattern_occurrences(text, DELIVERY_TERMS))
    features["delivery_flag"] = (features["delivery_term_count"] > 0).astype(int)

    features["negation_count"] = token_series.apply(lambda tokens: _count_lexicon_hits(tokens, NEGATION_TERMS))

    features["title_word_count"] = title_series.apply(_word_count)
    features["text_word_count"] = body_series.apply(_word_count)
    features["title_to_text_ratio"] = features["title_word_count"] / (features["text_word_count"] + 1.0)

    features["char_per_word"] = features["char_count"] / features["word_count"].clip(lower=1)
    features["exclamation_rate"] = features["exclamation_count"] / features["word_count"].clip(lower=1)
    features["question_rate"] = features["question_count"] / features["word_count"].clip(lower=1)
    features["caps_rate"] = features["caps_word_count"] / features["word_count"].clip(lower=1)

    features["asin_encoded"] = df["asin_encoded"].astype(int)
    features["asin_freq"] = df["asin_freq"].astype(float) if "asin_freq" in df.columns else 0.0

    features["sentiment_x_word_count"] = features["sentiment_score"] * features["word_count"]
    features["antithesis_norm_count"] = text_series.combine(features["word_count"], _antithesis_norm_count)

    return features

def get_numeric_feature_columns(mode: str = "full") -> list[str]:
    full_columns = [
        "word_count", "char_count", "exclamation_count", "question_count", "caps_word_count",
        "exclamation_rate", "question_rate", "caps_rate", "char_per_word",
        "sentiment_score", "positive_word_count", "negative_word_count", "sentiment_ratio",
        "mixed_sentiment_flag", "defect_term_count", "defect_flag", "price_term_count",
        "price_flag", "delivery_term_count", "delivery_flag", "negation_count",
        "asin_encoded", "asin_freq", "title_word_count", "text_word_count",
        "title_to_text_ratio", "sentiment_x_word_count", "antithesis_norm_count",
    ]

    if mode == "full":
        return full_columns
    if mode == "minimal":
        return ["word_count", "sentiment_score", "antithesis_norm_count", "asin_freq"]

    raise ValueError("mode deve ser 'full' ou 'minimal'")

In [ ]:
# Funções para medir desempenho e montar a pipeline completa do scikit-learn.
# A pipeline combina vetorização textual, transformação numérica e o classificador final.
# =========================
# AVALIAÇÃO E PIPELINE
# =========================

def evaluate_predictions(y_true: pd.Series, y_pred: pd.Series) -> dict[str, object]:
    labels = sorted(pd.unique(y_true))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_df = pd.DataFrame(
        cm,
        index=[f"real_{label}" for label in labels],
        columns=[f"pred_{label}" for label in labels],
    )

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "classification_report": classification_report(y_true, y_pred, digits=4),
        "confusion_matrix_text": cm_df.to_string(),
        "confusion_matrix_df": cm_df,
    }

def describe_train_patterns(train_df: pd.DataFrame, target_column: str) -> str:
    null_title = int(train_df["title"].isna().sum())
    null_text = int(train_df["text"].isna().sum())
    class_dist = train_df[target_column].value_counts().sort_index().to_dict()
    return (
        f"Amostras train={len(train_df)} | "
        f"title nulo={null_title} | text nulo={null_text} | "
        f"distribuição {target_column}={class_dist}"
    )

def build_estimator(
    model_name: str,
    random_state: int,
    svm_c: float = 1.0,
    svm_class_weight: str | None = None,
):
    model_name = model_name.lower()

    if model_name == "linear_svm":
        class_weight = None if svm_class_weight in (None, "none") else svm_class_weight
        return LinearSVC(
            C=svm_c,
            class_weight=class_weight,
            random_state=random_state,
            dual=False,
            max_iter=3000,
        )

    if model_name == "logreg":
        return LogisticRegression(
            max_iter=1500,
            solver="lbfgs",
            random_state=random_state,
        )

    if model_name == "lightgbm":
        try:
            from lightgbm import LGBMClassifier
        except ImportError as exc:
            raise ImportError("LightGBM não está instalado. Rode: pip install lightgbm") from exc

        return LGBMClassifier(
            objective="multiclass",
            num_class=5,
            n_estimators=350,
            learning_rate=0.05,
            random_state=random_state,
        )

    raise ValueError("model_name deve ser 'linear_svm', 'logreg' ou 'lightgbm'")

def build_training_pipeline(
    model_name: str,
    max_count_features: int,
    use_char_ngrams: bool,
    max_char_features: int,
    numeric_columns: list[str],
    random_state: int,
    svm_c: float = 1.0,
    svm_class_weight: str | None = None,
) -> Pipeline:

    transformers = [
        (
            "word_vectorizer",
            CountVectorizer(
                preprocessor=normalize_text_for_vectorizer,
                stop_words=get_portuguese_stopwords(),
                ngram_range=(1, 2),
                max_features=max_count_features,
                strip_accents="unicode",
                lowercase=True,
            ),
            "combined_text",
        ),
    ]

    if use_char_ngrams:
        transformers.append(
            (
                "char_vectorizer",
                CountVectorizer(
                    analyzer="char_wb",
                    ngram_range=(3, 5),
                    max_features=max_char_features,
                    lowercase=True,
                ),
                "combined_text",
            )
        )

    transformers.append(
        (
            "numeric",
            StandardScaler(with_mean=False),
            numeric_columns,
        )
    )

    preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
    estimator = build_estimator(
        model_name=model_name,
        random_state=random_state,
        svm_c=svm_c,
        svm_class_weight=svm_class_weight,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", estimator),
        ]
    )

## Antes de executar: funções mais importantes para entender

Estas são as funções-chave do notebook:

### `prepare_datasets(train_df, test_df)`
É a porta de entrada da preparação dos dados. Ela:
- limpa os campos textuais;
- cria `combined_text`;
- ajusta o codificador de `ASIN` usando apenas o treino;
- adiciona `asin_encoded` e `asin_freq`.

### `build_feature_frame(df)`
Transforma um DataFrame já limpo em outro DataFrame com:
- a coluna `combined_text`, que será vetorizada;
- as features numéricas manuais, que serão escalonadas e combinadas com o texto.

### `get_numeric_feature_columns(mode)`
Define quais colunas numéricas entram no modelo.
- `full`: usa mais sinais manuais;
- `minimal`: usa apenas um subconjunto mais compacto.

### `build_training_pipeline(...)`
Cria a pipeline final do `scikit-learn`, unindo:
- vetorização do texto;
- transformação das colunas numéricas;
- classificador.

### `evaluate_predictions(y_true, y_pred)`
Resume o desempenho do modelo em métricas e estruturas fáceis de inspecionar, como:
- F1-macro;
- accuracy;
- classification report;
- matriz de confusão.

> Entender bem essas cinco funções já permite compreender quase todo o fluxo do notebook.


## 1) Verificação dos arquivos de entrada

In [ ]:
for path in [TRAIN_PATH, TEST_PATH, SAMPLE_SUBMISSION_PATH]:
    print(path, "->", Path(path).exists())

In [ ]:
train_raw, test_raw = load_train_test(TRAIN_PATH, TEST_PATH, target_column=TARGET_COLUMN)

log("Dados carregados com sucesso.")
log(describe_train_patterns(train_raw, target_column=TARGET_COLUMN))

print("\nColunas do train:", train_raw.columns.tolist())
print("Colunas do test:", test_raw.columns.tolist())
print("\nAmostras do treino:")
display(train_raw.head(3))

## 2) Preprocessamento

O preprocessamento prepara os dados brutos para que o restante do pipeline funcione de forma consistente.

### O que acontece aqui
- preenchemos valores nulos em `title`, `text` e `ASIN`;
- removemos ruídos simples, como títulos contendo apenas `.`;
- criamos `combined_text`, que concatena título e corpo da review;
- ajustamos um codificador para `ASIN` com base no treino;
- calculamos a frequência relativa de cada produto no conjunto de treino.

### Por que `combined_text` é importante?
Em muitos problemas de review, tanto o título quanto o corpo trazem informação relevante. Unir os dois campos:
- simplifica o pipeline textual;
- evita manter dois vetorizadores separados;
- preserva contexto curto + contexto detalhado em uma única coluna.

### Por que usar `ASIN`?
`ASIN` identifica o produto. Mesmo sem usar embeddings complexos, podemos extrair dois sinais úteis:
- `asin_encoded`: uma codificação inteira do produto;
- `asin_freq`: quão frequente aquele produto aparece no treino.

Esses sinais podem capturar padrões específicos de produtos recorrentes.


In [ ]:
train_clean, test_clean, asin_encoder = prepare_datasets(train_raw, test_raw)

display(train_clean[["id", "ASIN", "title", "text", "combined_text", "asin_encoded", "asin_freq"]].head(3))

## 3) Construção das features

Nesta etapa transformamos os dados pré-processados em um conjunto de atributos que será consumido pela pipeline.

### Dois blocos de entrada
- **Texto:** a coluna `combined_text`, usada pelos vetorizadores.
- **Numéricas:** contagens, razões, sinais de sentimento e indicadores temáticos.

### Exemplos de features numéricas criadas
- **Comprimento do texto**
  - `word_count`, `char_count`
- **Pontuação e ênfase**
  - `exclamation_count`, `question_count`, `caps_word_count`
- **Polaridade lexical**
  - `sentiment_score`, `positive_word_count`, `negative_word_count`
- **Temas frequentes**
  - `defect_flag`, `price_flag`, `delivery_flag`
- **Sinais do produto**
  - `asin_encoded`, `asin_freq`

### Ideia central
Mesmo usando bag-of-words, algumas pistas simples podem ajudar o modelo:
- reviews muito curtas podem ter comportamento diferente;
- excesso de exclamações pode indicar emoção forte;
- menções a defeito, preço ou atraso podem empurrar a classe para notas mais baixas.

### Modos disponíveis
- `full`: usa o conjunto completo de features manuais;
- `minimal`: usa apenas um subconjunto mais enxuto e robusto.

Isso permite comparar **maior expressividade** versus **menor risco de ruído/overfitting**.


In [ ]:
train_features = build_feature_frame(train_clean)
test_features = build_feature_frame(test_clean)
numeric_columns = get_numeric_feature_columns(mode=FEATURE_MODE)

print("Shape train_features:", train_features.shape)
print("Shape test_features:", test_features.shape)
print("Quantidade de colunas numéricas selecionadas:", len(numeric_columns))
print("Primeiras colunas numéricas:", numeric_columns[:10])

display(train_features.head(3))

## 4) Construção do pipeline de treinamento

Agora juntamos tudo em uma única pipeline do `scikit-learn`.

### Como a pipeline é organizada
1. **Vetoriza o texto por palavras**
   - `CountVectorizer` com unigramas e bigramas.
2. **Opcionalmente vetoriza o texto por caracteres**
   - útil para lidar com erros ortográficos, variações de escrita e padrões sublexicais.
3. **Escalona as features numéricas**
   - com `StandardScaler(with_mean=False)`.
4. **Treina o classificador**
   - por padrão, `LinearSVC`.

### Por que usar `ColumnTransformer`?
Porque ele permite aplicar transformações diferentes em colunas diferentes:
- texto recebe vetorização;
- colunas numéricas recebem escalonamento.

Tudo isso dentro de um fluxo único e reproduzível.

### Por que `LinearSVC`?
Em classificação de textos com muitas features esparsas, modelos lineares costumam ser fortes baselines:
- treinam relativamente rápido;
- funcionam bem com bag-of-words;
- são robustos e fáceis de ajustar.


In [ ]:
# Construção efetiva da pipeline a partir das escolhas da seção de configuração.
# Neste ponto, ainda não treinamos o modelo; apenas definimos a arquitetura do experimento.
pipeline = build_training_pipeline(
    model_name=MODEL_NAME,
    max_count_features=COUNT_MAX_FEATURES,
    use_char_ngrams=USE_CHAR_NGRAMS,
    max_char_features=CHAR_MAX_FEATURES,
    numeric_columns=numeric_columns,
    random_state=RANDOM_STATE,
    svm_c=SVM_C,
    svm_class_weight=SVM_CLASS_WEIGHT,
)

pipeline

## 5) Validação cruzada opcional

Se `CV_FOLDS > 1`, executamos validação cruzada estratificada.

### O que isso significa?
O conjunto de treino é dividido em vários subconjuntos (*folds*). Em cada rodada:
- parte dos dados é usada para treino;
- uma parte diferente é usada para validação.

No final, calculamos a média e o desvio padrão das métricas.

### Por que isso é útil?
Um único split holdout pode dar uma estimativa instável. A validação cruzada:
- reduz dependência de uma única divisão;
- ajuda a comparar experimentos com mais confiança;
- fornece uma estimativa mais robusta de generalização.

### Métricas usadas
- **F1-macro**: importante quando queremos equilibrar desempenho entre as classes;
- **Accuracy**: mostra o acerto global.


In [ ]:
# A variável y contém a coluna-alvo do treino.
# A validação cruzada abaixo é opcional e serve para ter uma noção mais estável do desempenho médio.
y = train_raw[TARGET_COLUMN].astype(int)

if CV_FOLDS and CV_FOLDS > 1:
    log(f"Rodando validação cruzada estratificada com {CV_FOLDS} folds...")
    cv = StratifiedKFold(
        n_splits=CV_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    cv_result = cross_validate(
        pipeline,
        train_features,
        y,
        cv=cv,
        scoring={"f1_macro": "f1_macro", "accuracy": "accuracy"},
        n_jobs=-1,
        return_train_score=False,
    )

    cv_summary = pd.DataFrame({
        "fold": np.arange(1, CV_FOLDS + 1),
        "f1_macro": cv_result["test_f1_macro"],
        "accuracy": cv_result["test_accuracy"],
    })
    display(cv_summary)

    print("CV f1_macro médio:", round(cv_summary["f1_macro"].mean(), 5))
    print("CV f1_macro desvio:", round(cv_summary["f1_macro"].std(), 5))
    print("CV accuracy média:", round(cv_summary["accuracy"].mean(), 5))
    print("CV accuracy desvio:", round(cv_summary["accuracy"].std(), 5))
else:
    print("Validação cruzada desativada. Defina CV_FOLDS > 1 para ativar.")

## 6) Holdout estratificado para avaliação rápida

Mesmo quando usamos validação cruzada, ainda é útil fazer um holdout simples para inspecionar saídas de forma mais concreta.

### O que avaliamos aqui
- `f1_macro`
- `accuracy`
- relatório por classe
- matriz de confusão

### Por que estratificar?
A estratificação tenta manter a proporção das classes em treino e validação. Isso é importante para evitar:
- validação enviesada;
- ausência de classes raras em uma das partes;
- leitura distorcida das métricas.


In [ ]:
# Aqui fazemos um holdout estratificado.
# "Estratificado" significa preservar aproximadamente a proporção das classes em treino e validação.
X_train, X_val, y_train, y_val = train_test_split(
    train_features,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

log(f"Treinando modelo ({MODEL_NAME}) com split train/val...")
pipeline.fit(X_train, y_train)
val_preds = pipeline.predict(X_val)

metrics = evaluate_predictions(y_val, val_preds)

print("Validation f1_macro:", round(metrics["f1_macro"], 5))
print("Validation accuracy:", round(metrics["accuracy"], 5))
print("\nClassification report:")
print(metrics["classification_report"])

In [ ]:
print("Matriz de confusão:")
display(metrics["confusion_matrix_df"])

## 7) Refit em 100% do treino

Depois da análise no holdout, treinamos novamente usando todo o conjunto de treino.

### Por que fazer isso?
Durante a avaliação, deixamos uma parte dos dados separada para medir desempenho. Mas, na hora de gerar a submissão final, queremos aproveitar o máximo de informação disponível.

### Resultado desta etapa
- o modelo final é ajustado com todos os exemplos do treino;
- as predições são feitas sobre `test.csv`;
- um arquivo `submission.csv` é salvo no formato esperado pela competição.


In [ ]:
# Após avaliar a pipeline no holdout, treinamos novamente com 100% do conjunto de treino.
# Esse é o modelo usado para prever o conjunto de teste e gerar a submissão final.
log("Re-treinando em 100% do train para gerar submissão...")
pipeline.fit(train_features, y)
test_preds = pipeline.predict(test_features)

submission = pd.DataFrame({
    "id": test_raw["id"],
    TARGET_COLUMN: pd.Series(test_preds).astype(int),
})

if Path(SAMPLE_SUBMISSION_PATH).exists():
    expected_cols = pd.read_csv(SAMPLE_SUBMISSION_PATH, nrows=1).columns.tolist()
else:
    expected_cols = ["id", TARGET_COLUMN]

submission = submission[expected_cols]
submission.to_csv(SUBMISSION_PATH, index=False)

log(f"Arquivo de submissão salvo em: {SUBMISSION_PATH}")
display(submission.head())

## 8) Próximos experimentos recomendados

Este notebook também pode ser usado como base de experimentação.

### O que vale testar
- `FEATURE_MODE`: comparar `full` vs `minimal`;
- `SVM_C`: por exemplo `0.5`, `1.0`, `2.0`, `5.0`, `10.0`;
- `SVM_CLASS_WEIGHT`: `none` vs `balanced`;
- `USE_CHAR_NGRAMS`: ativado ou não;
- `COUNT_MAX_FEATURES` e `CHAR_MAX_FEATURES`.

### Estratégia recomendada
Mude **um fator por vez** sempre que possível. Isso facilita entender o impacto real de cada alteração.

### Sugestão prática
Monte um pequeno quadro de experimentos contendo:
- configuração usada;
- F1-macro;
- accuracy;
- observações.

Assim, o notebook deixa de ser apenas um script executável e passa a ser também um **registro de investigação**.


In [ ]:
# Pequena tabela de apoio para registrar experimentos manualmente dentro do notebook.
# Isso ajuda a comparar combinações de hiperparâmetros e métricas obtidas.
# Exemplo de tabela manual para registrar experimentos:
experiment_log = pd.DataFrame([
    {
        "model": MODEL_NAME,
        "feature_mode": FEATURE_MODE,
        "use_char_ngrams": USE_CHAR_NGRAMS,
        "count_max_features": COUNT_MAX_FEATURES,
        "char_max_features": CHAR_MAX_FEATURES,
        "svm_c": SVM_C,
        "svm_class_weight": SVM_CLASS_WEIGHT,
        "test_size": TEST_SIZE,
        "f1_macro_holdout": metrics["f1_macro"],
        "accuracy_holdout": metrics["accuracy"],
    }
])

display(experiment_log)